# Task 6: Hypothesis Testing 

In [1]:
import pandas as pd
import numpy as np

from scipy.stats import ttest_ind
from scipy.stats import levene
from scipy.stats import mannwhitneyu

df = pd.read_csv("../data/goldman_sachs_cleaned.csv")

df["TransactionDate"] = pd.to_datetime(df["TransactionDate"])

# Create Account Statistics

account_stats = (
    df.groupby("AccountID")
      .agg(
          AvgBalance=("AccountBalance", "mean"),
          TotalVolume=("TransactionAmount", "sum")
      )
      .reset_index()
)

account_stats.head()

,AccountID,AvgBalance,TotalVolume
0,Acc10117,70107.007957,199480.967430
1,Acc10996,43568.008084,250739.550950
2,Acc11062,38137.132610,27189.136160
3,Acc11188,69652.151044,257576.603590
4,Acc11285,97401.348560,96729.609841


In [2]:
# Calculating Median Transaction Volume

volume_median = account_stats["TotalVolume"].median()

print("Median Transaction Volume:", volume_median)

Median Transaction Volume: 201951.8756075


In [3]:
# Creating Volume Segments

account_stats["VolumeSegment"] = np.where(
    account_stats["TotalVolume"] >= volume_median,
    "High Volume",
    "Low Volume"
)

account_stats.head()

,AccountID,AvgBalance,TotalVolume,VolumeSegment
0,Acc10117,70107.007957,199480.967430,Low Volume
1,Acc10996,43568.008084,250739.550950,High Volume
2,Acc11062,38137.132610,27189.136160,Low Volume
3,Acc11188,69652.151044,257576.603590,High Volume
4,Acc11285,97401.348560,96729.609841,Low Volume


# Hypotheses

- Null Hypothesis (H₀):

There is no significant difference in the average account balance between High-Volume and Low-Volume transaction accounts.

- Alternative Hypothesis (H₁):

High-Volume transaction accounts have a significantly higher average account balance than Low-Volume transaction accounts.

- **Significance Level (α): 0.05**

In [4]:
# Separate High and Low Volume Accounts

high_volume = account_stats[
    account_stats["VolumeSegment"] == "High Volume"
]["AvgBalance"]

low_volume = account_stats[
    account_stats["VolumeSegment"] == "Low Volume"
]["AvgBalance"]

In [5]:
# Levene's Test

levene_stat, levene_p = levene(
    high_volume,
    low_volume
)

print("Levene Statistic:", levene_stat)
print("Levene p-value:", levene_p)

Levene Statistic: 5.813833416922785
Levene p-value: 0.016842716890132493


In [6]:
# Independent t-Test

t_stat, p_value = ttest_ind(
    high_volume,
    low_volume,
    equal_var=(levene_p > 0.05)
)

print("t-Statistic:", t_stat)
print("p-value:", p_value)

t-Statistic: -0.012000502597877487
p-value: 0.9904387756889178


In [7]:
# Interpret Result

alpha = 0.05

if p_value < alpha:
    print("Reject the Null Hypothesis (H₀)")
    print("High-volume accounts have significantly higher average balances.")
else:
    print("Fail to Reject the Null Hypothesis (H₀)")
    print("No significant difference exists between the two groups.")

Fail to Reject the Null Hypothesis (H₀)
No significant difference exists between the two groups.


In [8]:
# Mann-Whitney U Test (ADDITIONAL TEST FOR VALIDATION)

u_stat, u_p = mannwhitneyu(
    high_volume,
    low_volume,
    alternative="greater"
)

print("Mann-Whitney U Statistic:", u_stat)
print("p-value:", u_p)

Mann-Whitney U Statistic: 4712.0
p-value: 0.4928585484789359


# Hypothesis Test Interpretation

At Independent Samples, t-test was performed to compare the average account balances of High-Volume and Low-Volume transaction accounts.

Levene's Test was conducted first to determine whether equal variances could be assumed.

Additionally, a Mann-Whitney U Test was performed as a non-parametric validation.

The decision to reject or fail to reject the null hypothesis was based on a significance level of 0.05.

# Task 6 Summary

Hypothesis testing was conducted to determine whether High-Volume transaction accounts maintain significantly higher average balances than Low-Volume accounts.

Levene's Test was used to assess variance equality, followed by an Independent Samples t-test. A Mann-Whitney U Test was also performed to validate the findings.

# Key Findings

- Levene Test p-value: **0.0168**
- Independent t-Test p-value: **0.9904**
- Mann-Whitney Test p-value: **0.4929**
- The null hypothesis was **not rejected.**

# Insights

The analysis did not find statistical evidence that customers with higher transaction
volumes maintain higher average account balances. This indicates that transaction
frequency or volume alone may not be a reliable indicator of account balance.

